In [0]:
from pyspark.sql import functions as F

SILVER_TABLE       = "nyc_taxi.silver.yellow_trips"
GOLD_BOROUGH_MONTH = "nyc_taxi.gold.trips_by_borough_month"
GOLD_DAILY         = "nyc_taxi.gold.trips_daily"

silver = spark.table(SILVER_TABLE)

In [0]:
borough_month = (
    silver
    .withColumn("pickup_borough", F.coalesce("pickup_borough", F.lit("Unmatched")))
    .groupBy("pickup_year", "pickup_month", "pickup_borough")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("fare_amount").alias("total_fare"),
        F.sum("tip_amount").alias("total_tip"),
        F.sum("trip_distance").alias("total_distance"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
    )
)

(
    borough_month.write.format("delta").mode("overwrite")
    .saveAsTable(GOLD_BOROUGH_MONTH)
)

In [0]:
daily = (
    silver
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_revenue"),
        F.sum("passenger_count").alias("total_passengers"),
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
    )
)

(
    daily.write.format("delta").mode("overwrite")
    .saveAsTable(GOLD_DAILY)
)

In [0]:
%sql
COMMENT ON TABLE nyc_taxi.gold.trips_by_borough_month IS
  'Trips per pickup borough and month (2019-01..03). Additive measures (counts, sums) plus rounded averages. Null boroughs labelled Unmatched.';

COMMENT ON TABLE nyc_taxi.gold.trips_daily IS
  'Daily trip totals by pickup date (2019-01..03): counts, revenue, passengers, average duration and distance.';

In [0]:
silver_count = spark.table(SILVER_TABLE).count()
bm_total     = spark.table(GOLD_BOROUGH_MONTH).agg(F.sum("trip_count")).first()[0]
daily_total  = spark.table(GOLD_DAILY).agg(F.sum("trip_count")).first()[0]

print(f"silver rows            : {silver_count:,}")
print(f"gold borough_month sum : {bm_total:,}   match = {bm_total == silver_count}")
print(f"gold daily sum         : {daily_total:,}   match = {daily_total == silver_count}")
print(f"daily rows (days)      : {spark.table(GOLD_DAILY).count()}   (expected 90)")

In [0]:
display(spark.table(GOLD_BOROUGH_MONTH).orderBy("pickup_year", "pickup_month", F.desc("trip_count")))
display(spark.table(GOLD_DAILY).orderBy("pickup_date"))